# Python 課後整理：循環、大小寫交錯與矩陣操作

**上課日期：2026/08/29**

這份筆記依照課堂原始 notebook 整理，主要包含：

1. 用 `visited` 找出索引關係中的循環數量
2. 用 `.isupper()` / `.islower()` 判斷英文大小寫
3. 尋找固定長度、大小寫交錯的字串區段
4. 比較較直接的解法與較有效率的 run-length 解法
5. 矩陣翻轉、旋轉與「反向還原」的課堂草稿

> 原始 notebook 沒有附完整題目敘述、輸入限制與正式範例，因此這份整理只根據課堂留下的程式與筆記說明，不自行補出題目名稱或未出現的規則。


## 1. 用 `visited` 計算有幾個循環

課堂留下的索引資料：

```text
4 7 2 9 6 0 8 1 5 3
```

程式把它讀進：

```python
friend = [int(i) for i in input().split()]
```

可以把：

```python
friend[i]
```

理解成：

> 現在位於索引 `i`，下一步要走到索引 `friend[i]`。

假設資料是：

```python
friend = [4, 7, 2, 9, 6, 0, 8, 1, 5, 3]
```

從 `0` 開始：

```text
0 → 4 → 6 → 8 → 5 → 0
```

又回到已經走過的 `0`，所以形成一個循環。

其他循環為：

```text
1 → 7 → 1
2 → 2
3 → 9 → 3
```

因此這組資料共有 **4 個循環**。


In [ ]:
n = int(input())

friend = [int(i) for i in input().split()]

visited = [0] * n
count = 0

for i in range(n):

    if visited[i] == 0:

        count += 1
        current = i

        # 例如：0 → 4 → 6 → 8 → 5 → 0
        while visited[current] == 0:
            visited[current] = 1
            current = friend[current]

print(count)


### 1.1 `visited` 是做什麼的？

一開始：

```python
visited = [0] * n
```

例如 `n = 10`：

```text
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
```

可以把它理解成：

```text
0 → 還沒走過
1 → 已經走過
```

當程式走到某個位置：

```python
visited[current] = 1
```

就把它標記成「已走過」。

接著：

```python
current = friend[current]
```

前往下一個位置。

---

### 1.2 為什麼外層還要有 `for`？

```python
for i in range(n):
```

會依序檢查每個索引。

如果：

```python
visited[i] == 0
```

表示這個位置還不屬於前面已經走過的循環，因此找到了一個新的循環：

```python
count += 1
```

如果已經走過，就不需要再從那裡重新走一次。

### 執行概念

```text
找到一個沒走過的位置
        ↓
count + 1
        ↓
沿著 friend[current] 一直走
        ↓
每走一格就標成 visited
        ↓
碰到已經 visited 的位置
        ↓
這一圈結束
```


## 2. 判斷英文大小寫

原始 notebook 先留下了建立大小寫字母串列的草稿：

```text
capital = ['A', ...]
lower   = ['a', ...]
```

但後來的程式實際使用 Python 內建的：

```python
.isupper()
.islower()
```

因此不需要自己建立完整的 `A ~ Z` 與 `a ~ z` 串列。

### `.isupper()`

判斷字元是否為大寫。

### `.islower()`

判斷字元是否為小寫。


In [ ]:
'A'.islower()

In [ ]:
'a'.isupper()

上面兩個判斷都會得到：

```text
False
```

相對地：

```python
'A'.isupper()
'a'.islower()
```

則會得到 `True`。

後面的字串題就是利用這兩個方法判斷目前這一段是大寫還是小寫。


## 3. 固定長度的大小寫交錯區段

課堂筆記留下兩個例子：

```text
aaBBccDD
```

若：

```text
k = 2
```

可以分成：

```text
aa | BB | cc | DD
```

每一段：

- 長度都是 `2`
- 大寫、小寫交替出現

所以總長度為：

```text
2 + 2 + 2 + 2 = 8
```

另一個例子：

```text
aafAXbbCDCCC
```

依照連續大小寫先分段：

```text
aaf | AX | bb | CDCCC
 3     2    2      5
```

當 `k = 2` 時，課堂筆記取：

```text
af | AX | bb | CD
```

每段都取 `2` 個字元，因此：

```text
2 + 2 + 2 + 2 = 8
```

這也說明一件重要的事：

> 某一整段連續大小寫的長度可以大於 `k`，但有效答案仍可能從那一段的中間開始，或在那一段中提早結束。


### 3.1 第一種解法：從每一個起點往後找

原始 notebook 先寫出下面的版本：


In [ ]:
k = int(input())
word = input()

big = 0

for i in range(len(word)):

    ans = 0
    count = 0

    if word[i].isupper():
        check = 0
    else:
        check = 1

    for j in range(i, len(word)):

        if check == 0:

            if word[j].isupper():
                count += 1
            else:
                break

        else:

            if word[j].islower():
                count += 1
            else:
                break

        if count == k:

            ans += k
            count = 0

            check = 1 if check == 0 else 0

            if ans > big:
                big = ans

print(big)


### 3.2 重要變數

| 變數 | 用途 |
|---|---|
| `k` | 每一個大小寫區段需要的長度 |
| `word` | 輸入字串 |
| `i` | 這一次嘗試的起點 |
| `j` | 從起點向後檢查的位置 |
| `check` | 現在期待大寫還是小寫 |
| `count` | 目前這一小段已累積幾個字元 |
| `ans` | 這個起點目前找到的有效長度 |
| `big` | 到目前為止的最大有效長度 |

這裡的約定是：

```text
check = 0 → 期待大寫
check = 1 → 期待小寫
```

所以：

```python
if word[i].isupper():
    check = 0
else:
    check = 1
```

會先根據起點決定第一段應該檢查大寫還是小寫。


### 3.3 找滿 `k` 個後就切換大小寫

當：

```python
count == k
```

代表目前這一段已經剛好收集到 `k` 個字元。

因此：

```python
ans += k
count = 0
```

把有效長度增加 `k`，並重新計數。

接著：

```python
check = 1 if check == 0 else 0
```

切換下一段要找的大小寫：

```text
0 → 1
1 → 0
```

也就是：

```text
大寫 → 小寫
小寫 → 大寫
```

如果期待大寫卻碰到小寫，或期待小寫卻碰到大寫：

```python
break
```

這一次從 `i` 開始的搜尋就停止。


## 4. 為什麼還要改進演算法？

課堂筆記寫下：

```text
algo → 好壞（效能）
10 min → 好的：1000 runs
         壞的：10 runs
```

這裡的重點不是這兩個數字本身，而是：

> **同一個問題，不同演算法可能有非常大的速度差距。**

上一個版本：

```python
for i in range(len(word)):
    ...
    for j in range(i, len(word)):
```

有一層 `for` 裡面再做另一層向後搜尋。

當字串變長時，需要重複檢查很多字元。

課堂接著把問題改成：

> 先把「連續同樣大小寫」的長度整理出來，再處理這些長度。

這樣就不必從每一個字元重新開始搜尋。


## 5. 改進版：先做 run-length 分段

原始改進版程式：


In [ ]:
k = int(input())
s = input()

runs = []
count = 1

for i in range(1, len(s)):

    if s[i].isupper() == s[i-1].isupper():
        count += 1
    else:
        runs.append(count)
        count = 1

runs.append(count)

current = 0
best = 0

for length in runs:

    if length < k:
        current = 0

    elif length == k:
        current += 1

        if current > best:
            best = current

    else:
        if current + 1 > best:
            best = current + 1

        current = 1

print(best * k)


### 5.1 第一階段：先記錄每一段連續大小寫的長度

核心判斷：

```python
s[i].isupper() == s[i-1].isupper()
```

不是在比較兩個字元內容相不相同，而是在比較：

> 這兩個字元是不是同樣都屬於大寫，或同樣都不是大寫。

如果相同：

```python
count += 1
```

表示還在同一個 run。

如果不同：

```python
runs.append(count)
count = 1
```

表示前一段結束，開始新的大小寫區段。

最後還要：

```python
runs.append(count)
```

把最後一段補進去。

### 例子

```text
aaBBccDD
```

會得到：

```python
runs = [2, 2, 2, 2]
```

而：

```text
aafAXbbCDCCC
```

會得到：

```python
runs = [3, 2, 2, 5]
```


### 5.2 第二階段：比較每一段長度和 `k`

程式分成三種情況。

#### 情況一：`length < k`

```python
if length < k:
    current = 0
```

這一段連 `k` 個字元都湊不到，因此目前連續的有效區段被中斷。

---

#### 情況二：`length == k`

```python
elif length == k:
    current += 1
```

這一整段剛好可以完整使用。

如果連續出現很多個長度等於 `k` 的 run：

```text
k, k, k, k
```

`current` 就會一路增加。

---

#### 情況三：`length > k`

```python
else:
```

這個 run 比 `k` 長。

從課堂例子：

```text
aaf | AX | bb | CDCCC
```

可以看出，長 run 仍可能提供一個長度為 `k` 的區段：

```text
af | AX | bb | CD
```

因此程式會讓它貢獻一個區段，但不能把整個長 run 當成多個完整交錯區段。

最後：

```python
print(best * k)
```

因為 `best` 記錄的是最多連續幾個有效區段，而每個區段長度都是 `k`。


## 6. 矩陣操作與反向還原

這一部分在原始 notebook 中仍是**課堂草稿**，程式尚未完成。

筆記先記下：

```text
A → 1 → 2 → 3 → B
```

如果要從最後的 `B` 找回原本的 `A`，操作順序必須反過來：

```text
B → 3 的反操作 → 2 的反操作 → 1 的反操作 → A
```

這是很重要的概念：

> **要復原一連串操作，要從最後一個操作開始，依反方向逐步還原。**

原始 notebook 沒有留下完整題目規則，因此以下只整理課堂已經畫出的矩陣變化。


### 6.1 上下翻轉

課堂筆記：

```text
1 2          5 6
3 4    →     3 4
5 6          1 2
```

這是把「列的順序」上下顛倒：

```text
第 1 列 ↔ 第 3 列
第 2 列留在中間
```

對串列來說，課堂也測試了：

```python
lst[::-1]
```

它可以把串列順序反轉。


In [ ]:
lst = [0, 1, 2]
print(lst[::-1])

輸出：

```text
[2, 1, 0]
```

同樣的概念若用在「矩陣的列」上，就可以用來思考上下翻轉。

另外，上下翻轉再做一次：

```text
原矩陣
  ↓ 翻轉
上下顛倒
  ↓ 再翻轉
原矩陣
```

因此這種翻轉的反操作就是它自己。


### 6.2 90° 旋轉：順時針與逆時針要分清楚

原始筆記中出現了兩個方向。

#### 逆時針 90°

```text
1 2 3          3 6
4 5 6    →     2 5
               1 4
```

#### 順時針 90°

原 notebook 另一個草稿寫成：

```text
1 2 3          4 1
4 5 6    →     5 2
               6 3
```

所以：

```text
順時針 90° 的反操作 = 逆時針 90°
逆時針 90° 的反操作 = 順時針 90°
```

如果原本是一個：

```text
2 × 3
```

的矩陣，旋轉 90° 後會變成：

```text
3 × 2
```

也就是列數與欄數會交換。


### 6.3 原始未完成程式草稿

課堂最後開始寫：

```python
R, C, M = input().split()
R = int(R)
C = int(C)
M = int(M)

m = []

for i in range(R):
    n = input()
    m.append(n)

lst = []
new_m = []

for i in range(R):
    for j in range(C):
        if M == 0:
            lst.append(m[i][j])
        else:
            # 尚未完成
```

原 notebook 在 `else:` 之後就停止，因此這一段還不是完整、可執行的解答。

這裡不自行補完，避免把課堂尚未完成的想法誤寫成老師最後採用的演算法。

目前可以確定已經開始處理的概念有：

- `R`：列數
- `C`：欄數
- `M`：某個操作相關的輸入值
- `m`：儲存輸入矩陣
- `lst`、`new_m`：準備存放轉換後資料

但 `M == 0` 與 `else` 的完整規則，單靠這份 notebook 無法確定。


## 7. 課後快速複習

### 小測驗

1. 在循環題中，為什麼遇到 `visited[i] == 1` 時不用重新走一次？

2. 對：

   ```python
   friend = [4, 7, 2, 9, 6, 0, 8, 1, 5, 3]
   ```

   從 `3` 開始會形成哪一個循環？

3. `.isupper()` 與 `.islower()` 分別判斷什麼？

4. 若：

   ```text
   s = aaBBccDD
   k = 2
   ```

   run-length 串列 `runs` 會是多少？

5. 為什麼 `length < k` 時：

   ```python
   current = 0
   ```

6. `lst[::-1]` 對：

   ```python
   [1, 2, 3, 4]
   ```

   會得到什麼？

7. 如果矩陣原本是 `2 × 3`，旋轉 90° 後尺寸會變成多少？

### 本堂重點

- `visited` 可以避免在圖式／索引關係中重複走訪同一個循環。
- `while` 適合用來沿著「下一個位置」一直走到停止條件成立。
- `.isupper()` / `.islower()` 可以直接判斷英文大小寫。
- 從每個起點重新搜尋雖然直觀，但可能重複做很多工作。
- 先整理成 `runs`，可以把逐字元問題轉成「每一段長度」的問題。
- 好的演算法可以大幅減少重複計算。
- 復原多個操作時，要倒著做每個操作的反操作。
- 90° 旋轉後，矩陣的列數與欄數會交換。
